# Global XAI For Already-Trained Best Neural Model

This Colab notebook does **not train anything**. It loads the already trained best neural checkpoint, computes global XAI across all 23 MM-IMDb genre labels, and saves every artifact needed by the local dashboard's **Global XAI - Best Neural** tab.

Required files in the project checkout or mounted Drive:

- `dataset/data/multimodal_imdb.hdf5`
- `dataset/data/metadata.npy`
- `outputs/splits/test_indices.npy`
- `outputs/models/best/neural_multimodal_best.pt`

## 1. Mount Drive And Open Project

Edit `COLAB_PROJECT_DIR` only if your repository lives somewhere else on Google Drive.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import urllib.request

IN_COLAB = 'google.colab' in sys.modules
COLAB_PROJECT_DIR = Path('/content/drive/MyDrive/Explaining-Predictions-of-Machine-Learning-Models')
GITHUB_REPO = 'https://github.com/SamuelSulan/Explaining-Predictions-of-Machine-Learning-Models.git'
RAW_BASE_URL = 'https://raw.githubusercontent.com/SamuelSulan/Explaining-Predictions-of-Machine-Learning-Models/main'
UPDATE_GLOBAL_XAI_FILES_FROM_GITHUB = True


def download_text_file(url: str, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url) as response:
        target.write_bytes(response.read())
    print('Updated:', target)


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not COLAB_PROJECT_DIR.exists():
        COLAB_PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', GITHUB_REPO, str(COLAB_PROJECT_DIR)], check=True)
    os.chdir(COLAB_PROJECT_DIR)
else:
    os.chdir(Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd())

PROJECT_ROOT = Path.cwd()
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

if UPDATE_GLOBAL_XAI_FILES_FROM_GITHUB:
    # Update only the files needed by this notebook. This avoids `git pull` conflicts
    # and does not touch dataset/model files in Drive.
    download_text_file(f'{RAW_BASE_URL}/src/mmimdb/global_xai.py', SRC_PATH / 'mmimdb' / 'global_xai.py')
    download_text_file(f'{RAW_BASE_URL}/scripts/run_global_xai.py', PROJECT_ROOT / 'scripts' / 'run_global_xai.py')

required_source = SRC_PATH / 'mmimdb' / 'global_xai.py'
if not required_source.exists():
    raise FileNotFoundError(f'Missing {required_source}. The global XAI source file is required for this notebook.')

print('Project root:', PROJECT_ROOT)
print('Global XAI source:', required_source)


## 2. Install Missing Runtime Dependencies

This keeps Colab's CUDA-enabled PyTorch installation intact and only installs missing extras.

In [ ]:
import importlib.util

missing = []
for module, package in [
    ('captum', 'captum==0.9.0'),
    ('iterstrat', 'iterative-stratification==0.1.9'),
]:
    if importlib.util.find_spec(module) is None:
        missing.append(package)

if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

## 3. Configure Global XAI Run

`RUN_MODE = 'full'` is intended for final thesis artifacts. Change it to `'smoke'` for a quick check that paths and model loading work.

In [ ]:
RUN_MODE = 'full'  # 'full' for final artifacts, 'smoke' for a quick check

CONFIG_PATH = 'configs/default.yaml'
CHECKPOINT_PATH = 'outputs/models/best/neural_multimodal_best.pt'
OUTPUT_DIR = 'outputs/global_xai/best_neural'
SPLIT = 'test'
SEED = 42

if RUN_MODE == 'smoke':
    SAMPLES_PER_LABEL = 1
    BATCH_SIZE = 4
    TOKEN_CANDIDATES_PER_LABEL = 5
    TOKEN_OCCLUSION_TOP_K = 2
    IMAGE_OCCLUSION_GRID = 2
else:
    SAMPLES_PER_LABEL = 25
    BATCH_SIZE = 16
    TOKEN_CANDIDATES_PER_LABEL = 20
    TOKEN_OCCLUSION_TOP_K = 10
    IMAGE_OCCLUSION_GRID = 4

ENABLE_TOKEN_OCCLUSION = True
ENABLE_IMAGE_OCCLUSION = True

# Optional: if local per-instance XAI exists, the notebook also aggregates saved Layer IG token attributions.
LOCAL_XAI_DIR = 'outputs/colab_test_xai_all_models/run_newest_balanced/xai/best_neural'
GENERATE_LOCAL_XAI_IF_EMPTY = True
LOCAL_XAI_LIMIT = 23 if RUN_MODE == 'full' else 3
LOCAL_XAI_N_STEPS = 8 if RUN_MODE == 'full' else 2

print({
    'RUN_MODE': RUN_MODE,
    'CHECKPOINT_PATH': CHECKPOINT_PATH,
    'OUTPUT_DIR': OUTPUT_DIR,
    'SAMPLES_PER_LABEL': SAMPLES_PER_LABEL,
    'TOKEN_OCCLUSION_TOP_K': TOKEN_OCCLUSION_TOP_K,
    'IMAGE_OCCLUSION_GRID': IMAGE_OCCLUSION_GRID,
    'GENERATE_LOCAL_XAI_IF_EMPTY': GENERATE_LOCAL_XAI_IF_EMPTY,
    'LOCAL_XAI_LIMIT': LOCAL_XAI_LIMIT,
})

## 4. Verify Required Files

This cell fails early if the already-trained checkpoint or dataset files are missing. There is intentionally no training fallback.

In [ ]:
from mmimdb.data import DatasetPaths
from mmimdb.utils import load_config, resolve_path

config = load_config(CONFIG_PATH)
paths = DatasetPaths.from_config(config)
required_paths = {
    'HDF5 dataset': paths.hdf5,
    'metadata': paths.metadata,
    'train split': resolve_path(config['splits']['output_dir']) / 'train_indices.npy',
    'validation split': resolve_path(config['splits']['output_dir']) / 'val_indices.npy',
    'test split': resolve_path(config['splits']['output_dir']) / 'test_indices.npy',
    'trained best neural checkpoint': resolve_path(CHECKPOINT_PATH),
}

missing_paths = {name: path for name, path in required_paths.items() if not Path(path).exists()}
if missing_paths:
    for name, path in missing_paths.items():
        print(f'MISSING {name}: {path}')
    raise FileNotFoundError('Required files are missing. Upload/copy them to Drive first; this notebook does not train a replacement model.')

for name, path in required_paths.items():
    print(f'OK {name}: {path}')

## 5. Methods Saved For Thesis

The run writes `global_xai_methods_for_thesis.csv` with concise descriptions you can reuse in the thesis.

In [ ]:
import pandas as pd
from mmimdb.global_xai import METHOD_DESCRIPTIONS

pd.DataFrame(METHOD_DESCRIPTIONS)

## 6. Prepare Local XAI Source For Aggregated Layer IG Tokens

Aggregated Layer Integrated Gradients token summaries need local per-instance neural XAI `explanation.json` files. This cell checks `LOCAL_XAI_DIR`; if it is missing or empty and `GENERATE_LOCAL_XAI_IF_EMPTY = True`, it creates the folder and generates a small local XAI set from the already-trained checkpoint. No training is performed.


In [ ]:
from mmimdb.data import load_labels
from mmimdb.splits import load_split_indices
from mmimdb.xai import explain_neural_samples
import numpy as np

local_xai_path = Path(LOCAL_XAI_DIR)
local_xai_path.mkdir(parents=True, exist_ok=True)
existing_local_explanations = list(local_xai_path.rglob('explanation.json'))
print('Local XAI directory:', local_xai_path.resolve())
print('Existing local explanations:', len(existing_local_explanations))


def choose_balanced_indices(test_indices, labels, limit, seed=42):
    rng = np.random.default_rng(seed)
    selected = []
    seen = set()
    label_order = list(rng.permutation(labels.shape[1]))
    for label_i in label_order:
        positives = [int(idx) for idx in test_indices if labels[int(idx), label_i] == 1 and int(idx) not in seen]
        if positives:
            idx = positives[0]
            selected.append(idx)
            seen.add(idx)
        if len(selected) >= int(limit):
            break
    if len(selected) < int(limit):
        for idx in test_indices:
            idx = int(idx)
            if idx not in seen:
                selected.append(idx)
                seen.add(idx)
            if len(selected) >= int(limit):
                break
    return np.asarray(selected, dtype=np.int64)


if not existing_local_explanations and GENERATE_LOCAL_XAI_IF_EMPTY:
    print('LOCAL_XAI_DIR is empty. Generating local neural XAI explanations for Layer IG aggregation...')
    train_idx, val_idx, test_idx = load_split_indices(resolve_path(config['splits']['output_dir']))
    labels_all = load_labels(paths.hdf5)
    local_indices = choose_balanced_indices(test_idx, labels_all, LOCAL_XAI_LIMIT, SEED)
    print('Local XAI indices:', local_indices.tolist())
    local_summary = explain_neural_samples(
        CHECKPOINT_PATH,
        paths.hdf5,
        paths.metadata,
        local_indices,
        local_xai_path,
        target_policy='predicted',
        target_top_k=3,
        max_targets_per_sample=3,
        ensure_at_least_one_target=True,
        top_k=12,
        n_steps=LOCAL_XAI_N_STEPS,
        measure_performance=True,
        enable_layer_integrated_gradients_text=True,
        enable_integrated_gradients_image=False,
        enable_gradcam=False,
        enable_modality_ablation=False,
        enable_modality_shapley=False,
        enable_token_occlusion=False,
        enable_image_occlusion=False,
        enable_experimental_set_explanation=False,
    )
    existing_local_explanations = list(local_xai_path.rglob('explanation.json'))
    print('Generated local explanations:', len(existing_local_explanations))
elif not existing_local_explanations:
    print('LOCAL_XAI_DIR is empty and GENERATE_LOCAL_XAI_IF_EMPTY is False. Aggregated Layer IG tokens will be skipped.')
else:
    print('Using existing local XAI explanations for aggregated Layer IG tokens.')

LOCAL_XAI_DIR_FOR_GLOBAL = str(local_xai_path) if existing_local_explanations else None


## 7. Run Global XAI

This loads the trained model from `CHECKPOINT_PATH`, computes all configured global XAI methods, and saves tables/figures/JSON under `OUTPUT_DIR`.

In [ ]:
from mmimdb.global_xai import run_global_neural_xai

summary = run_global_neural_xai(
    config_path=CONFIG_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    output_dir=OUTPUT_DIR,
    split=SPLIT,
    samples_per_label=SAMPLES_PER_LABEL,
    seed=SEED,
    batch_size=BATCH_SIZE,
    token_candidates_per_label=TOKEN_CANDIDATES_PER_LABEL,
    token_occlusion_top_k=TOKEN_OCCLUSION_TOP_K,
    image_occlusion_grid=IMAGE_OCCLUSION_GRID,
    enable_token_occlusion=ENABLE_TOKEN_OCCLUSION,
    enable_image_occlusion=ENABLE_IMAGE_OCCLUSION,
    local_xai_dir=LOCAL_XAI_DIR_FOR_GLOBAL,
)

print('Global XAI summary:', Path(OUTPUT_DIR, 'global_xai_summary.json').resolve())
print('Selected samples:', summary['selected_sample_count'])
print('Runtime seconds:', round(summary['runtime_seconds'], 2))
print('Warnings:', len(summary['warnings']))

## 8. Inspect Saved Data

In [ ]:
from IPython.display import Image, display

out = Path(OUTPUT_DIR)
expected_outputs = [
    'global_xai_summary.json',
    'global_xai_methods_for_thesis.csv',
    'prediction_context.csv',
    'modality_ablation_and_permutation.csv',
    'global_modality_shapley.csv',
    'global_token_occlusion.csv',
    'aggregated_lig_tokens.csv',
    'per_label_image_occlusion_heatmaps.csv',
]

for name in expected_outputs:
    path = out / name
    print(('OK   ' if path.exists() else 'MISS ') + str(path))

display(pd.read_csv(out / 'prediction_context.csv').head(23))
display(pd.read_csv(out / 'modality_ablation_and_permutation.csv'))

for figure_name in [
    'global_modality_utilization.png',
    'per_label_modality_utilization.png',
    'global_token_occlusion_top.png',
    'prediction_context.png',
]:
    path = out / figure_name
    if path.exists():
        display(Image(filename=str(path)))

## 9. Package Artifacts

The dashboard can read the folder directly. This zip is convenient if you want to download or copy the global XAI outputs elsewhere.

In [ ]:
import shutil

zip_base = Path(OUTPUT_DIR).with_name(Path(OUTPUT_DIR).name + '_artifacts')
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=Path(OUTPUT_DIR))
print('Packaged artifacts:', zip_path)

if IN_COLAB:
    print('The zip is saved in Drive. You can also download it from the Colab file browser if needed.')